<h1>Catatan Studi Regresi Logistik & Softmax</h1>
  
  <p>
    Regresi Logistik adalah algoritma terawasi yang digunakan untuk klasifikasi biner.
    Alih-alih memprediksi nilai kontinu, ia memprediksi probabilitas suatu masukan
    <em>x</em> termasuk dalam sebuah kelas.
  </p>
  


  ### Fungsi Sigmoid
    Elemen kuncinya adalah fungsi sigmoid (logistik), yang didefinisikan sebagai:

  $\sigma(z) = \frac{1}{1+e^{-z}}$

  dimana $z = \mathbf{w}^T\mathbf{x} + b$.
  


### Kerugian Lintas Entropi
Untuk contoh pelatihan dengan label benar $y$ dan prediksi $\hat{y}=\sigma(z)$, kerugiannya adalah:
$L(\hat{y}, y) = -\bigl[ y\log(\hat{y}) + (1-y)\log(1-\hat{y}) \bigr]$
Sasaran selama pelatihan adalah meminimalkan kerugian rata-rata pada kumpulan data.

In [ ]:

### Contoh Python: Regresi Logistik dengan scikit-learn
  
#1. Menggunakan scikit-learn untuk melatih regresi logistik pada masalah biner (subset Iris)
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#2. Muat kumpulan data Iris dan pilih dua kelas untuk klasifikasi biner
data = load_iris()
X = data.data[data.target != 2]
y = data.target[data.target != 2]

#3. Pisahkan data menjadi set pelatihan dan pengujian
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size=0.2, 
    random_state=42
)

#4. Buat model regresi logistik dan latih
model = LogisticRegression(solver='lbfgs', C=1.0, max_iter=200)
model.fit(X_train, y_train)

#5. Memprediksi dan mencetak akurasi
y_pred = model.predict(X_test)
print("Akurasi Regresi Logistik:", accuracy_score(y_test, y_pred))
  


In [ ]:
  
### Contoh Python: Regresi Logistik dari Awal (Gradien Descent)

#Kode Python berikut mengimplementasikan regresi logistik secara manual menggunakan NumPy.
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def komputasi_biaya(X, y, w, b):
    m = X.shape[0]
    z = np.dot(X, w) + b
    a = sigmoid(z)
    # Tambahkan nilai kecil (epsilon) untuk menghindari log(0)
    epsilon = 1e-15
    biaya = -np.mean(y * np.log(a + epsilon) + (1 - y) * np.log(1 - a + epsilon))
    return biaya

def gradien_descent(X, y, w, b, kecepatan_belajar, jumlah_iterasi):
    m = X.shape[0]
    for i in range (jumlah_iterasi):
        z = np.dot(X, w) + b
        a = sigmoid(z)
        dw = np.dot(X.T, (a - y)) / m
        db = np.sum(a - y) / m
        
        w = w - kecepatan_belajar * dw
        b = b - kecepatan_belajar * db
        
        if i % 1000 == 0:
            biaya = komputasi_biaya(X, y, w, b)
            print(f"Iterasi {i}, Biaya: {biaya}")
    return w, b

In [ ]:
# Buat kumpulan data klasifikasi biner sintetis
np.random.seed(0)
X = np.random.randn(100, 2)
# Tetapkan aturan sederhana: jika jumlah fitur > 0, kelas 1; lain, kelas 0.
y = (np.sum(X, axis=1) > 0).astype(int)

# Inisialisasi parameter
w = np.zeros(X.shape[1])
b = 0

# Latih regresi logistik menggunakan penurunan gradien
w, b = gradien_descent(X, y, w, b, kecepatan_belajar=0.1, jumlah_iterasi=10000)
print("Bobot yang dipelajari:", w, "Bias:", b)

# Prediksi data pelatihan
z = np.dot(X, w) + b
prediksi = (sigmoid(z) >= 0.5).astype(int)
akurasi = np.mean(prediksi == y)
print("Akurasi Pelatihan (dari awal):", akurasi)


### Regresi Softmax menggeneralisasi regresi logistik untuk mengklasifikasikan input lebih dari dua kelas.
### Fungsi Softmax 
Untuk vektor logit
$$
\mathbf{z} = [z_1, z_2, \dots, z_K] 
$$

fungsi softmaxnya adalah:
$$
\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}
$$


In [ ]:
# Contoh Python: Regresi Softmax dengan scikit-learn
from sklearn.linear_model import LogisticRegression

# Muat dataset Iris lengkap (3 kelas)
X, y = load_iris(return_X_y=True)

# Melatih regresi logistik multinomial (regresi softmax)
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=200)
model.fit(X, y)

# Memprediksi dan menghitung akurasi pada data pelatihan
y_pred = model.prediksi(X)
print("Akurasi Regresi Softmax:", accuracy_score(y, y_pred))

Untuk menangkap hubungan non-linier, kita dapat menambahkan fitur polinomial ke data masukan.
Misalnya, untuk satu fitur $x$, Anda dapat menggunakan $x^2$, $x^3$, dll. Untuk beberapa fitur
fitur, istilah interaksi seperti $x_1x_2$ dapat ditambahkan.
    Catatan: Jumlah fitur meningkat secara kombinatorial seiring dengan derajat polinomial.
    Dengan $N$ ciri asli dan derajat $k$, jumlah suku diberikan oleh:
$\binom{N+k}{k}$

In [ ]:
  
# Contoh Python: Menggunakan PolynomialFeatures dengan scikit-learn
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Buat saluran yang mengubah fitur dan kemudian menerapkan regresi logistik.
degree = 2 # Ubah derajat sesuai kebutuhan
model_poly = make_pipeline(
    PolynomialFeatures(degree=degree),
    LogisticRegression(solver='lbfgs', max_iter=200)
)

# Gunakan subset biner yang sama dari dataset Iris
model_poly.fit(X_train, y_train)
y_pred_poly = model_poly.predict(X_test)
print("Akurasi Regresi Logistik Polinomial (derajat={}):".format(degree),
      accuracy_score(y_test, y_pred_poly))

  <p>
    Hyperparameter umum meliputi:
  </p>
  <ul>
    <li><strong>Kekuatan Regularisasi (C atau λ):</strong> Mengontrol overfitting. <em>C</em> yang lebih kecil (atau λ yang lebih besar) berarti regularisasi yang lebih kuat.</li>
    <li><strong>Jenis Penalti:</strong> L1 (mendorong ketersebaran) atau L2 (memberi penalti pada beban besar).</li>
    <li><strong>Pemecah:</strong> Algoritma seperti <code>liblinear</code>, <code>lbfgs</code>, <code>sag</code>, atau <code>saga</code>.</li>
    <li><strong>max_iter:</strong> Iterasi maksimum untuk konvergensi.</li>
    <li><strong>tol:</strong> Toleransi terhadap kriteria pemberhentian.</li>
  </ul>
  <p>
    Pengaturan ini disesuaikan (menggunakan metode seperti pencarian grid) untuk mengoptimalkan kinerja.
  </p>
  
  <h2>Ringkasan</h2>
  <p>
    Dengan menggabungkan teori dengan contoh kode praktis, Anda dapat membangun dasar yang kuat dalam:
  </p>
  <ul>
    <li>Regresi Logistik Biner menggunakan fungsi sigmoid dan kerugian lintas entropi.</li>
    <li>Memperluas masalah kelas jamak dengan Regresi Softmax.</li>
    <li>Meningkatkan kapasitas model dengan Fitur Polinomial (dengan memperhatikan risiko overfitting dan peningkatan komputasi).</li>
    <li>Menyesuaikan hyperparameter seperti regularisasi, pemecah, dan batas iterasi.</li>
  </ul>
  
  <p>
    Contoh kode Python menggambarkan implementasi berbasis perpustakaan (menggunakan scikit-learn) dan
    implementasi manual menggunakan NumPy dengan penurunan gradien. Bereksperimenlah dengan contoh-contoh ini untuk memperdalam pemahaman Anda.
  </p>
  
  </tubuh>
</html>